Распознавание базы рукописных цифр MNIST с использованием CNN

Необходимо построить систему распознавания рукописных цифр с использованием базы MNIST на основе Сверточной нейронной сети. Вероятность правильного распознавания на тестовой базе должна быть больше 99%

In [1]:
import tensorflow as tf
import numpy as np

print("TensorFlow version:", tf.__version__)

# 1. Загрузка базы MNIST
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 2. Нормализация значений пикселей к диапазону [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# 3. Добавление размерности канала: (60000, 28, 28) -> (60000, 28, 28, 1)
x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]

print("Размерность обучающей выборки:", x_train.shape)
print("Размерность тестовой выборки:", x_test.shape)

TensorFlow version: 2.21.0
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Размерность обучающей выборки: (60000, 28, 28, 1)
Размерность тестовой выборки: (10000, 28, 28, 1)


In [2]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),

    # Первый свёрточный блок
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(0.25),

    # Второй свёрточный блок
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(0.25),

    # Полносвязный классификатор
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10)  # Выходной слой логитов (10 классов цифр от 0 до 9)
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 26, 26, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 26, 26, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 13, 13, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 11, 11, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 272,490 (1.04 MB)

 Trainable params: 271,850 (1.04 MB)

 Non-trainable params: 640 (2.50 KB)

In [3]:
predictions = model(x_train[:1]).numpy()
print("Логиты до обучения:", predictions)
print("Вероятности (Softmax):", tf.nn.softmax(predictions).numpy())

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
print("Начальное значение функции потерь:", loss_fn(y_train[:1], predictions).numpy())

Логиты до обучения: [[-0.00068478  0.00518303  0.01687236  0.0405525   0.07732629 -0.02725371
   0.03314863  0.0060903  -0.08847364 -0.00090952]]
Вероятности (Softmax): [[0.09923045 0.09981443 0.10098804 0.10340799 0.10728148 0.09662873
  0.1026452  0.09990502 0.09089055 0.09920815]]
Начальное значение функции потерь: 2.3368793


In [4]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=loss_fn,
    metrics=['accuracy']
)

# Автоматическое снижение шага обучения при выходе на плато
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1
)

# Обучение модели
history = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=10,
    validation_data=(x_test, y_test),
    callbacks=[lr_scheduler]
)

# Финальная проверка на тестовой выборке
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nИтоговая точность на тестовой базе: {test_acc * 100:.2f}%")

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 258s 533ms/step - accuracy: 0.9358 - loss: 0.2121 - val_accuracy: 0.8253 - val_loss: 0.5024 - learning_rate: 0.0010
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 253s 515ms/step - accuracy: 0.9792 - loss: 0.0681 - val_accuracy: 0.9903 - val_loss: 0.0274 - learning_rate: 0.0010
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 246s 524ms/step - accuracy: 0.9844 - loss: 0.0511 - val_accuracy: 0.9915 - val_loss: 0.0264 - learning_rate: 0.0010
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 254s 507ms/step - accuracy: 0.9869 - loss: 0.0416 - val_accuracy: 0.9929 - val_loss: 0.0224 - learning_rate: 0.0010
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 242s 516ms/step - accuracy: 0.9891 - loss: 0.0359 - val_accuracy: 0.9941 - val_loss: 0.0184 - learning_rate: 0.0010
Epoch 6/10
330/469 ━━━━━━━━━━━━━━━━━━━━ 1:08 492ms/step - accuracy: 0.9904 - loss: 0.0334

KeyboardInterrupt: 

In [16]:
# Предсказания для всех 10 000 тестовых изображений
all_probs = probability_model.predict(x_test, verbose=0)
all_preds = np.argmax(all_probs, axis=1)

# Поиск индексов, где предсказание НЕ совпало с истинной меткой
errors = np.where(all_preds != y_test)[0]
print(f"Всего ошибок: {len(errors)} из {len(y_test)}")

# Вывод первых 5 ошибок
print("Индексы с ошибками: ", errors[:5])
print("Модель предсказала: ", all_preds[errors[:5]])
print("На самом деле это:  ", y_test[errors[:5]])

Всего ошибок: 86 из 10000
Индексы с ошибками:  [259 449 571 740 846]
Модель предсказала:  [0 5 9 9 9]
На самом деле это:   [6 3 4 4 7]
